### New algorithm for enumerating every $\Z_2^n$-colorable PL~spheres of Picard number $5$

In [118]:
using Oscar
using ProgressBars
using SparseArrays
using Serialization


In [181]:
function boundary_incidence_facets_to_ridges(facets::Vector{Vector{Int}})
    d = length(facets[1]) - 1  # facet dimension
    @assert all(length(f) == d+1 for f in facets) "Need a pure complex (all facets same size)."

    # collect ridges (each facet contributes its (d-1)-subfaces by deleting one vertex)
    ridge_dict = Dict{Vector{Int}, Int}()  # ridge -> row index
    ridges = Vector{Vector{Int}}()
    for f in facets
        for k in eachindex(f)
            r = [f[i] for i in eachindex(f) if i != k]  # already sorted since f is sorted
            if !haskey(ridge_dict, r)
                push!(ridges, r)
                ridge_dict[r] = length(ridges)
            end
        end
    end

    m = length(ridges)
    n = length(facets)

    # build sparse 0/1 matrix (m x n): ridges x facets
    I = Int[]   # row indices
    J = Int[]   # col indices
    V = Int8[]  # values (all ones)

    for (j, f) in pairs(facets)
        for k in eachindex(f)
            r = [f[i] for i in eachindex(f) if i != k]
            i = ridge_dict[r]
            push!(I, i)
            push!(J, j)
            push!(V, Int8(1))
        end
    end

    A = Bool.(sparse(I, J, V, m, n))  # SparseMatrixCSC{Int8}

    return ridges, A
end



boundary_incidence_facets_to_ridges (generic function with 1 method)

In [120]:
function kernel_basis_mod2(A)
    m, n = size(A)

    # Build R = A mod 2 as a BitMatrix
    R = falses(m, n)
    @inbounds for j in 1:n
        for p in nzrange(A, j)
            i = rowvals(A)[p]
            if isodd(A.nzval[p])
                R[i, j] ⊻= true
            end
        end
    end

    # Reduced row echelon form over GF(2), recording pivot columns and pivot rows
    pivcol = Int[]
    pivrow = Int[]
    row = 1
    @inbounds for col in 1:n
        row > m && break

        # find pivot in this column at/under current row
        piv = 0
        for r in row:m
            if R[r, col]
                piv = r
                break
            end
        end
        piv == 0 && continue

        # swap into position
        if piv != row
            @views begin
                tmp = copy(R[row, :])
                R[row, :] .= R[piv, :]
                R[piv, :] .= tmp
            end
        end

        push!(pivcol, col)
        push!(pivrow, row)

        # eliminate this column in all other rows (RREF)
        @views for r in 1:m
            if r != row && R[r, col]
                R[r, :] .= R[r, :] .!= R[row, :]   # XOR over GF(2)
            end
        end

        row += 1
    end

    # free columns = variables you set to 1 to generate kernel basis vectors
    pivot_set = Set(pivcol)
    free_cols = [j for j in 1:n if !(j in pivot_set)]

    basis = BitVector[]
    isempty(free_cols) && return basis  # full column rank over GF(2)

    # Build one kernel vector per free column
    @inbounds for f in free_cols
        x = falses(n)
        x[f] = true

        # back-substitute using RREF rows: x[pivot] = sum(other entries in row) mod 2
        for t in length(pivcol):-1:1
            c = pivcol[t]
            r = pivrow[t]
            @views begin
                # parity of dot(R[r, :], x) excluding pivot c
                tmp = R[r, :] .& x
                tmp[c] = false
                x[c] = isodd(count(tmp))
            end
        end

        push!(basis, x)
    end

    return basis
end


kernel_basis_mod2 (generic function with 1 method)

In [203]:
using SparseArrays

"""
    kernel_elements_with_Ax_in_0_or_2_noprune(A, basis) -> Vector{BitVector}

Full enumeration WITHOUT pruning.

- A is strictly 0/1 and used over ℤ for Ax.
- basis is a GF(2)-basis of ker(A mod 2), given as Vector{BitVector}.
- Enumerates all nonzero x in the GF(2)-span of `basis` and returns those with Ax ∈ {0,2}^m.
- Complexity: Θ(2^k * (cost to update/check)), where k = length(basis).
"""
function kernel_elements_with_Ax_in_0_or_2(
    A::SparseMatrixCSC{<:Integer},
    basis::Vector{BitVector},
)
    m, n = size(A)
    k = length(basis)
    println("Dimension of the kernel: ",k)
    @assert all(length(b) == n for b in basis) "Basis vectors must have length = #cols(A)."

    # column -> incident rows (A is 0/1)
    rv = rowvals(A)
    colrows = Vector{Vector{Int}}(undef, n)
    for j in 1:n
        colrows[j] = [rv[p] for p in nzrange(A, j)]
    end

    # supports of basis vectors (columns they toggle)
    supp = Vector{Vector{Int}}(undef, k)
    for t in 1:k
        supp[t] = findall(basis[t])
    end

    x = falses(n)
    counts = zeros(Int16, m)  # Ax over ℤ
    sols = BitVector[]

    function toggle_basis!(t::Int)
        @inbounds for j in supp[t]
            turning_on = !x[j]
            x[j] = turning_on
            δ = turning_on ? Int16(1) : Int16(-1)
            for i in colrows[j]
                counts[i] += δ
            end
        end
    end

    @inline function feasible_final()::Bool
        @inbounds for i in 1:m
            c = counts[i]
            if !(c == 0 || c == 2)
                return false
            end
        end
        return true
    end

    function dfs(t::Int, any_one::Bool)
        if t > k
            if any_one && feasible_final()
                push!(sols, copy(x))
            end
            return
        end

        # branch 0: do not include basis[t]
        dfs(t + 1, any_one)

        # branch 1: include basis[t]
        toggle_basis!(t)
        dfs(t + 1, true)
        toggle_basis!(t)  # toggle back (self-inverse over GF(2))
    end

    dfs(1, false)  # excludes zero vector by any_one=false
    return sols
end


kernel_elements_with_Ax_in_0_or_2

In [122]:
# If homology is UNREDUCED in your setup (most common):
function is_homology_sphere(K::Oscar.SimplicialComplex)
    b = betti_numbers(K)
    d = dim(K)
    for i in 0:d
        if i == 0 || i == d
            b[i+1] == 1 || return false
        else
            b[i+1] == 0 || return false
        end
    end
    return true
end


is_homology_sphere (generic function with 1 method)

In [168]:
using Polymake
const topaz = Polymake.topaz
using ProgressMeter
using Nemo
F = GF(2)


# Build canonical vertex list present in a matroid (sorted)
function vertex_list_of_bases(bases::Vector{Vector{Int}})
    s = Set{Int}()
    for b in bases
        for v in b
            push!(s, v)
        end
    end
    return sort(collect(s))
end

# Construct a matrix from a binary matroid

function binary_matrix_representation(M; B=nothing)
    is_binary(M) || error("Matroid is not binary")

    # ground set
    E = collect(matroid_groundset(M))
    n = length(E)

    # choose basis
    if B === nothing
        B = first(bases(M))
    else
        length(B) == rank(M) || error("Provided set is not a basis")
    end

    r = length(B)

    # index maps
    row_of = Dict(b => i for (i,b) in enumerate(B))
    col_of = Dict(e => j for (j,e) in enumerate(E))

    # initialize matrix
    A = matrix(F,zeros(Int, (r, n)))

    # basis columns = identity
    for b in B
        A[row_of[b], col_of[b]] = F(1)
    end

    # non-basis columns via fundamental circuits
    for e in E
        e in B && continue
        C = fundamental_circuit(M, B, e)
        for b in C
            b in B || continue
            A[row_of[b], col_of[e]] = F(1)
        end
    end

    return A, B, E
end

# ---------------------------
# using Topaz for isom
# ---------------------------

function bases_to_topaz_complex(bases::Vector{Vector{Int}})
    # collect and sort vertices
    verts = sort!(unique(vcat(bases...)))

    # map vertex labels to 0-based indices
    vmap = Dict(v => i-1 for (i,v) in enumerate(verts))

    facets = [ [vmap[x] for x in B] for B in bases ]
    return topaz.SimplicialComplex(FACETS = facets), verts
end

function topaz_isomorphism(basesA, basesB)
    SC_A, vertsA = bases_to_topaz_complex(basesA)
    SC_B, vertsB = bases_to_topaz_complex(basesB)
    length(vertsA) == length(vertsB) || return nothing
    T = topaz.find_facet_vertex_permutations(SC_A, SC_B)
    if T===nothing
        return nothing
    end
    _,p=T
    perm = Dict(
        vertsA[i] => vertsB[p[i]+1]
        for i in eachindex(vertsA)
    )
    return perm
end






# ---------------------------
# Main builder
# ---------------------------
"""
build_iso_db!(Iso_DB, mat_DB; ms = sort(collect(keys(mat_DB))), verbose=false)

- Iso_DB will be mutated (create empty Dict before passing).
- mat_DB: Dict{Int, Vector{Vector{Vector{Int}}}} mapping m -> list of matroid-bases
- For each m in ms (skips if m-1 not present), and each k in mat_DB[m],
  finds first vertex v in 1:m such that corank(contract_at_vertex(M,v)) >= corank(M),
  computes contraction, then finds l and a permutation mapping contraction -> some mat_DB[m-1][l].
- Stores Iso_DB[m][k] = (l, mapping::Dict{Int,Int}) or (-1, nothing) if not found.
"""
function build_iso_db!(Iso_DB::Dict{Int,Dict{Int,Tuple{Int,Int,Any}}}, mat_DB::Dict{Int,Vector{Vector{Vector{Int}}}}; ms=nothing, verbose=false)
    # ms === nothing && (ms = sort(collect(keys(mat_DB))))
    for m in ms
        if !haskey(mat_DB, m-1)
            verbose && @info("Skipping m=$m because mat_DB[$(m-1)] not present")
            continue
        end
        Iso_DB[m] = Dict{Int,Tuple{Int,Any}}()
        @showprogress desc="Computing m=$(m)..." for (k, Mbases) in enumerate(mat_DB[m])
            V=vertex_list_of_bases(Mbases)
            M = matroid_from_bases(Mbases,V)
            found_v = nothing
            Mv_chosen = nothing
            # pick smallest v in 1:m satisfying corank(contract) >= corank(M)
            v_chosen=nothing
            for v in V
                if v in coloops(M)
                    continue
                end
                Mv = deletion(M,v)
                if rank(Mv)==rank(M)
                    found_v=true
                    v_chosen=v
                    Mv_chosen=Mv
                    break
                end
            end
            if found_v === nothing
                v_chosen = V[1]
            end
            Mv_chosen = deletion(M,v_chosen)
            deletion_bases = bases(Mv_chosen)

            # search through mat_DB[m-1] for an isomorphism
            perm = nothing
            found_index = -1
            found_isom = false
            for (l, target_bases) in enumerate(mat_DB[m-1])
                M2 = matroid_from_bases(target_bases,vertex_list_of_bases(target_bases))
                if !is_isomorphic(Mv_chosen,M2)
                    continue
                end
                perm = topaz_isomorphism(target_bases,deletion_bases)
                if perm !== nothing
                    found_index = l
                    break
                end
            end

            if perm === nothing
                Iso_DB[m][k] = (-1, v_chosen, nothing)
                A,_,_ = binary_matrix_representation(M)
                display(A)
                verbose && @warn(
                    "No isomorphic matroid found for contraction  m=$m, k=$k at v=$v_chosen"
                )
            else
                Iso_DB[m][k] = (found_index,v_chosen, perm)
                # verbose && @info(
                #     "Found iso: m=$m, k=$k -> m-1 index=$found_index, v=$v_chosen"
                # )
            end
        end
    end
    return Iso_DB
end


build_iso_db!

In [158]:
iso_DB = Dict{Int,Dict{Int,Tuple{Int,Int,Any}}}()

build_iso_db!(iso_DB,mat_DB,ms=8:13,verbose=true)

# for m=8:11
#     for data in iso_DB[m]
#         k,perm = data
#         println(perm)
#     end
# end

open("rank_5_iso_DB_m_9-13.jls", "w") do io
    serialize(io, iso_DB)
end

Computing m=8... 100%|███████████████████████████████████| Time: 0:00:00
Computing m=9... 100%|███████████████████████████████████| Time: 0:00:00
Computing m=10... 100%|██████████████████████████████████| Time: 0:00:03
Computing m=11... 100%|██████████████████████████████████| Time: 0:00:32
Computing m=12... 100%|██████████████████████████████████| Time: 0:03:44
Computing m=13... 100%|██████████████████████████████████| Time: 0:19:24


In [178]:
using SparseArrays

"""
    enumerate_kernel_Ax_0or2_with_mandatory_facets(A, basis; mandatory_facets, forbid_implied=true)

Enumerate all x in span_GF(2)(basis) ⊆ {0,1}^n such that:
1) x[j]=1 for all j in `mandatory_facets`,
2) Ax over ℤ has every entry in {0,2}.

Speedups:
- Deterministic propagation from the 0/2 row-sum constraint to infer additional forced
  mandatory/forbidden facets (sound preprocessing).
- Linear reduction: restrict the GF(2) kernel span to an affine subspace satisfying the
  (mandatory + forbidden) facet constraints via GF(2) elimination on coefficient space.

Inputs:
- A: m×n sparse 0/1 ridge×facet incidence matrix (SparseMatrixCSC)
- basis: Vector{BitVector} (length n each), a GF(2)-basis of ker(A mod 2)
- mandatory_facets: Vector{Int} (facet indices forced to 1)
- forbid_implied: if true, uses propagation to infer forbidden facets too

Output:
- Vector{BitVector} of all solutions x (each length n)

Notes:
- No pruning in the final enumeration step (full enumeration of the reduced affine space).
- Throws error if the facet constraints are inconsistent with the kernel span.
"""
function enumerate_kernel_Ax_0or2_with_mandatory_facets(
    A::SparseMatrixCSC{<:Integer},
    basis::Vector{BitVector};
    mandatory_facets::BitVector,
    forbid_implied::Bool = true,
)
    m, n = size(A)
    k = length(basis)
    @assert all(length(b) == n for b in basis) "Each basis vector must have length n=#cols(A)."
    @assert length(mandatory_facets)==n
    # ---------- adjacency: col->rows and row->cols ----------
    rv = rowvals(A)
    colrows = Vector{Vector{Int}}(undef, n)
    for j in 1:n
        colrows[j] = [rv[p] for p in nzrange(A, j)]
    end
    rowcols = [Int[] for _ in 1:m]
    for j in 1:n
        for i in colrows[j]
            push!(rowcols[i], j)
        end
    end

    # ---------- preprocessing propagation (0/2 row sums) ----------
    # state: -1 forbidden, 0 unknown, +1 mandatory
    state = fill(Int8(0), n)
    state[mandatory_facets] .= 1

    function propagate_0or2!()
        # queue rows affected by assignments: initialize with rows touched by initial mandatory facets
        inqueue = fill(false, m)
        queue = Int[]
        for j in mandatory_facets
            for i in colrows[j]
                if !inqueue[i]
                    push!(queue, i); inqueue[i] = true
                end
            end
        end
        # If no mandatory facets, still do nothing (no forced info).
        isempty(queue) && return true

        while !isempty(queue)
            i = pop!(queue)
            inqueue[i] = false

            cols = rowcols[i]
            s = 0
            u = 0
            last_unknown = 0
            @inbounds for j in cols
                if state[j] == 1
                    s += 1
                elseif state[j] == 0
                    u += 1
                    last_unknown = j
                end
            end

            if s > 2
                return false
            elseif s == 2
                # force all other incident facets to 0
                @inbounds for j in cols
                    if state[j] == 0
                        state[j] = -1
                        # enqueue rows affected by setting facet j
                        for r in colrows[j]
                            if !inqueue[r]
                                push!(queue, r); inqueue[r] = true
                            end
                        end
                    elseif state[j] == 1
                        # ok
                    end
                end
            elseif s == 1
                if u == 0
                    return false
                elseif u == 1
                    # forced 1
                    j = last_unknown
                    if state[j] == -1
                        return false
                    end
                    state[j] = 1
                    for r in colrows[j]
                        if !inqueue[r]
                            push!(queue, r); inqueue[r] = true
                        end
                    end
                end
            else # s == 0
                if u == 1
                    # cannot achieve 2 with a single unknown -> forced 0
                    j = last_unknown
                    if state[j] == 1
                        return false
                    end
                    state[j] = -1
                    for r in colrows[j]
                        if !inqueue[r]
                            push!(queue, r); inqueue[r] = true
                        end
                    end
                end
            end
        end

        return true
    end

    if forbid_implied
        ok = propagate_0or2!()
        ok || return BitVector[]
    end

    mand = findall(==(Int8(1)), state)
    forb = findall(==(Int8(-1)), state)

    # ---------- GF(2) affine restriction on coefficient space y ----------
    # x = sum_t y[t]*basis[t] (mod2). Constraints: x[j]=rhs_j for j in mand∪forb.
    C = vcat(mand, forb)
    rhs = BitVector(vcat(fill(true, length(mand)), fill(false, length(forb))))
    r = length(C)

    # Build constraint matrix Mmat[r,k] with Mmat[ri,t] = basis[t][C[ri]]
    Mmat = falses(r, k)
    for (ri, j) in enumerate(C)
        @inbounds for t in 1:k
            Mmat[ri, t] = basis[t][j]
        end
    end

    # Solve Mmat*y = rhs over GF(2): RREF
    function solve_affine_gf2(M::BitMatrix, b::BitVector)
        rr, kk = size(M)
        A2 = copy(M)
        b2 = copy(b)

        pivot_rows = Int[]
        pivot_cols = Int[]

        row = 1
        for col in 1:kk
            row > rr && break
            piv = 0
            for rrr in row:rr
                if A2[rrr, col]
                    piv = rrr; break
                end
            end
            piv == 0 && continue

            if piv != row
                @views begin
                    tmp = copy(A2[row, :]); A2[row, :] .= A2[piv, :]; A2[piv, :] .= tmp
                end
                b2[row], b2[piv] = b2[piv], b2[row]
            end

            push!(pivot_rows, row)
            push!(pivot_cols, col)
            @views for rrr in 1:rr
                if rrr != row && A2[rrr, col]
                    A2[rrr, :] .= A2[rrr, :] .!= A2[row, :]
                    b2[rrr] = xor(b2[rrr], b2[row])
                end
            end
            row += 1
        end

        for rrr in 1:rr
            if !any(@view A2[rrr, :]) && b2[rrr]
                error("Inconsistent facet constraints inside kernel span (no solution).")
            end
        end

        pivot_set = Set(pivot_cols)
        free_cols = [c for c in 1:kk if !(c in pivot_set)]

        y0 = falses(kk)
        for (pr, pc) in zip(pivot_rows, pivot_cols)
            y0[pc] = b2[pr]
        end

        yfree = BitVector[]
        for f in free_cols
            y = falses(kk)
            y[f] = true
            for (pr, pc) in zip(pivot_rows, pivot_cols)
                y[pc] = A2[pr, f]
            end
            push!(yfree, y)
        end

        return y0, yfree
    end

    y0, yfree = solve_affine_gf2(Mmat, rhs)

    # combine basis vectors according to coefficient vector y
    function combine_x(y::BitVector)
        x = falses(n)
        @inbounds for t in 1:k
            if y[t]
                x .= x .!= basis[t]  # XOR for Bool vectors
            end
        end
        return x
    end

    x0 = combine_x(y0)
    gens = [combine_x(u) for u in yfree]  # reduced generators in x-space

    # ---------- final enumeration in the reduced affine space, checking Ax in {0,2} ----------
    # initialize counts = A*x0 over ℤ
    counts = zeros(Int16, m)
    for j in findall(x0)
        @inbounds for i in colrows[j]
            counts[i] += Int16(1)
        end
    end
    # early reject
    for i in 1:m
        counts[i] > 2 && return BitVector[]
    end

    # generator supports (columns toggled)
    supp = [findall(g) for g in gens]
    tgens = length(gens)

    x = copy(x0)
    sols = BitVector[]

    function toggle_gen!(t)
        @inbounds for j in supp[t]
            turning_on = !x[j]
            x[j] = turning_on
            δ = turning_on ? Int16(1) : Int16(-1)
            for i in colrows[j]
                counts[i] += δ
            end
        end
    end

    function feasible_final()
        @inbounds for i in 1:m
            c = counts[i]
            if !(c == 0 || c == 2)
                return false
            end
        end
        return true
    end

    function dfs(t::Int)
        if t > tgens
            if feasible_final()
                push!(sols, copy(x))
            end
            return
        end
        dfs(t + 1)
        toggle_gen!(t)
        dfs(t + 1)
        toggle_gen!(t)
    end

    dfs(1)
    return sols
end


enumerate_kernel_Ax_0or2_with_mandatory_facets

In [198]:
function solve_gf2_constraint(A::AbstractMatrix{Bool}, b::Vector{Bool})
    """
    Résout A * x = b (mod 2) et retourne (solution_particulière, base_espace_nul)
    """
    m, n = size(A)
    
    # Matrice augmentée [A | b]
    M = hcat(copy(A), reshape(b, :, 1))
    
    pivot_cols = Int[]
    row = 1
    
    # Réduction de Gauss-Jordan
    for col in 1:n
        # Chercher un pivot
        pivot_row = findfirst(M[row:end, col])
        
        if pivot_row === nothing
            continue
        end
        
        pivot_row += row - 1
        
        # Échanger
        if pivot_row != row
            M[row, :], M[pivot_row, :] = M[pivot_row, :], M[row, :]
        end
        
        push!(pivot_cols, col)
        
        # Éliminer
        for i in 1:m
            if i != row && M[i, col]
                M[i, :] .⊻= M[row, :]
            end
        end
        
        row += 1
        if row > m
            break
        end
    end
    
    # Vérifier la cohérence
    for i in row:m
        if M[i, end]
            return nothing, nothing  # Système incompatible
        end
    end
    
    # Solution particulière
    particular = zeros(Bool, n)
    for (i, col) in enumerate(pivot_cols)
        particular[col] = M[i, end]
    end
    
    # Base de l'espace nul
    free_cols = setdiff(1:n, pivot_cols)
    num_free = length(free_cols)
    
    free_basis = zeros(Bool, n, num_free)
    for (j, free_col) in enumerate(free_cols)
        free_basis[free_col, j] = 1
        for (i, pivot_col) in enumerate(pivot_cols)
            free_basis[pivot_col, j] = M[i, free_col]
        end
    end
    
    return particular, free_basis
end

function enumerate_all_kernel_elements(A::SparseMatrixCSC{Bool}, fixed_positions::BitVector)
    """
    Énumère TOUS les vecteurs v tels que :
    - Chaque coefficient de A * v est exactement 0 ou 2 (dans ℤ)
    - v[i] = 1 pour tout i où fixed_positions[i] = 1
    - v[i] est libre (0 ou 1) pour tout i où fixed_positions[i] = 0
    
    Optimisé pour matrices creuses ÉNORMES (A reste creuse).
    """
    
    fixed_indices = findall(fixed_positions)
    
    # 1. Calculer une base du noyau mod 2
    # ATTENTION: pour très grandes matrices, il faut une méthode creuse pour le noyau
    # Ici on convertit temporairement juste pour le noyau (petite matrice)
    N = nullspace_mod2(Matrix(A))
    n, k = size(N)
    
    if k == 0
        return Vector{Bool}[]
    end
    
    # Pré-calculer A * N[:, j] pour chaque colonne (A reste creuse!)
    m = size(A, 1)
    A_N_cols = Vector{Vector{Int}}(undef, k)
    
    for j in 1:k
        v_j = Int.(N[:, j])
        # Produit matrice creuse * vecteur dense (efficace!)
        A_N_cols[j] = Vector(A * v_j)  # Convertir SparseVector en Vector
    end
    
    if isempty(fixed_indices)
        all_elements = Vector{Bool}[]
        N_cols = [N[:, j] for j in 1:k]
        
        for bits in 0:(2^k - 1)
            v = zeros(Bool, n)
            result = zeros(Int, m)
            
            for j in 1:k
                if (bits >> (j-1)) & 1 == 1
                    v .⊻= N_cols[j]
                    result .+= A_N_cols[j]  # Addition de vecteurs denses
                end
            end
            
            if all(x -> x == 0 || x == 2, result)
                push!(all_elements, v)
            end
        end
        
        return all_elements
    end
    
    # 2. Résoudre les contraintes
    constraint_matrix = N[fixed_indices, :]
    target = ones(Bool, length(fixed_indices))
    
    particular, free_basis = solve_gf2_constraint(constraint_matrix, target)
    
    if particular === nothing
        return Vector{Bool}[]
    end
    
    # 3. Énumérer toutes les solutions
    num_free = size(free_basis, 2)
    N_cols = [N[:, j] for j in 1:k]
    
    all_elements = Vector{Bool}[]
    sizehint!(all_elements, 2^num_free)
    
    for bits in 1:(2^num_free - 1)
        coeffs = copy(particular)
        for i in 1:num_free
            if (bits >> (i-1)) & 1 == 1
                coeffs .⊻= free_basis[:, i]
            end
        end
        
        v = zeros(Bool, n)
        result = zeros(Int, m)
        
        # Calcul incrémental (A reste creuse!)
        for j in 1:k
            if coeffs[j]
                v .⊻= N_cols[j]
                result .+= A_N_cols[j]
            end
        end
        if all(x->x == 0)
            continue
        end
        if all(x -> x == 0 || x == 2, result)
            push!(all_elements, v)
        end
    end
    
    return all_elements
end

enumerate_all_kernel_elements (generic function with 3 methods)

In [204]:
function enumerate_polygons(facets_list)
    ridges, A = boundary_incidence_facets_to_ridges(facets_list)
    basis = kernel_basis_mod2(A)
    mandatory=[1]
    return kernel_elements_with_Ax_in_0_or_2(A,basis)
end



enumerate_polygons (generic function with 1 method)

In [205]:
global mat_DB = open("rank_5_simple_bin_mat_DB.jls", "r") do io
    deserialize(io)
end

global iso_DB = open("rank_5_iso_DB_m_8-13.jls", "r") do io
    deserialize(io)
end

function subset_bitvector(superset::Vector{Vector{Int}}, subset::Vector{Vector{Int}})
    n = length(superset)
    result = falses(n)
    
    j = 1  # index dans subset
    for i in 1:n
        if j <= length(subset) && superset[i] == subset[j]
            result[i] = true
            j += 1
        end
    end
    
    return result
end

function initialize_DB!(pseudo_manifolds_DB::Dict{Int,Vector{Vector{Bool}}},mat_DB::Dict{Int,Vector{Vector{Vector{Int}}}})
    mmin = minimum(collect(keys(mat_DB)))
    pseudo_manifolds_DB[mmin] = [enumerate_polygons(bases) for bases in mat_DB[mmin]]
    return pseudo_manifolds_DB
end


function build_finalDB_single_v!(pseudo_manifolds_DB::Dict{Int,Vector{Vector{BitVector}}},mat_DB::Dict{Int,Vector{Vector{Vector{Int}}}},iso_DB::Dict{Int,Dict{Int,Tuple{Int,Int,Any}}},mmax)
    mmin = minimum(collect(keys(mat_DB)))
    for m=mmin:mmax
        pseudo_manifolds_DB[m] = Vector{Vector{BitVector}}()
        for (l,bases) in enumerate(mat_DB[m])
            ridges, A = boundary_incidence_facets_to_ridges(bases)
            push!(pseudo_manifolds_DB[m], Vector{BitVector}())
            if m==mmin
                mandatory_facets_bit=falses(length(bases))
                all_solutions_bit = enumerate_polygons(A)
                push!(pseudo_manifolds_DB[m][l],all_solutions_bit...)
            else
                index_contraction, v_contract, perm = iso_DB[m][l]
                for L in pseudo_manifolds_DB[m-1][index_contraction]
                    println(L)
                    # mandatory_facets = [sort(push([perm[i] for i in facet_L],v_contract)) for facet_L in mat_DB[m-1][index_contraction][findall(L)]]
                    # print(mandatory_facets)
                    # mandatory_facets_bit = subset_bitvector(bases, mandatory_facets)

                    # all_solutions_bit = enumerate_all_kernel_elements(A,mandatory_facets_bit)
                    # push!(pseudo_manifolds_DB[m][l],all_solutions_bit...)
                end
            end
        end
    end

end

global pseudo_manifolds_DB = Dict{Int,Vector{Vector{BitVector}}}()

build_finalDB_single_v!(pseudo_manifolds_DB,mat_DB,iso_DB,8)



error in running finalizer: ErrorException("task switch not allowed from inside gc finalizer")
ijl_error at /cache/build/tester-amdci4-12/julialang/julia-release-1-dot-11/src/rtutils.c:43
ijl_switch at /cache/build/tester-amdci4-12/julialang/julia-release-1-dot-11/src/task.c:635
try_yieldto at ./task.jl:948
wait at ./task.jl:1022
#wait#733 at ./condition.jl:130
wait at ./condition.jl:125 [inlined]
slowlock at ./lock.jl:157
lock at ./lock.jl:147 [inlined]
close at ./iostream.jl:42
jfptr_close_49882.1 at /home/vallee/.julia/juliaup/julia-1.11.6+0.x64.linux.gnu/lib/julia/sys.so (unknown line)
run_finalizer at /cache/build/tester-amdci4-12/julialang/julia-release-1-dot-11/src/gc.c:303
jl_gc_run_finalizers_in_list at /cache/build/tester-amdci4-12/julialang/julia-release-1-dot-11/src/gc.c:393
run_finalizers at /cache/build/tester-amdci4-12/julialang/julia-release-1-dot-11/src/gc.c:439
jl_mutex_unlock at /cache/build/tester-amdci4-12/julialang/julia-release-1-dot-11/src/julia_locks.h:80 [inli

LoadError: MethodError: no method matching boundary_incidence_facets_to_ridges(::SparseMatrixCSC{Bool, Int64})
The function `boundary_incidence_facets_to_ridges` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  boundary_incidence_facets_to_ridges([91m::Vector{Vector{Int64}}[39m)
[0m[90m   @[39m [35mMain[39m [90m[4mIn[181]:1[24m[39m
